# Solutions · 12. Gaussian mixtures and the Gibbs phase rule

Worked solutions to the three exercises of [Lecture 12](../../notebooks/part4_clustering/12_gaussian_mixture_models_and_gibbs_phase_rule.ipynb). Each solution is computed here and checked with `assert` statements against an exact identity or an independent calculation. The first code cell sets up the notebook and repeats the demonstrations of the lecture without printing them, because the solutions use their variables.

In [1]:
try:                      # use the installed package, or the copy in this repository
    import engml_2_applied
except ImportError:
    import pathlib, subprocess, sys
    for _root in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_root / "engml_2_applied" / "__init__.py").exists():
            sys.path.insert(0, str(_root))
            break
    else:                 # on Google Colab: install from GitHub
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "git+https://github.com/ktwyw/engml-2-applied.git"], check=True)
    import engml_2_applied

import inspect
import numpy as np
import matplotlib.pyplot as plt
from engml_2_applied import (clustering_phases, constants, descriptors, dim_reduction, linear_physics,
                             trees_ensembles, uncertainty)
from engml_2_applied.plotting import panel_label, use_style

use_style()

def show_source(obj):
    """Return the source code of a library function or class."""
    return inspect.getsource(obj)

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

import io, sys
_stdout, sys.stdout = sys.stdout, io.StringIO()   # the demonstrations are repeated without their output
heading("Demonstration 1: Expectation-Maximization GMM & Soft Coexistence Responsibilities")
t_c = np.linspace(0.0, 2.0 * np.pi, 40, endpoint=False)
disk = 0.35 * np.column_stack([np.cos(t_c), np.sin(t_c)])
X_phase1 = disk + np.array([-1.5, -1.0])
X_phase2 = disk + np.array([1.5, 1.0])
X_all = np.vstack([X_phase1, X_phase2])

gmm = clustering_phases.gmm_em_fit(X_all, k=2, seed=42)
resp = gmm["responsibilities"]

# Evaluate soft responsibility at the exact phase coexistence midpoint [0.0, 0.0]:
x_mid = np.array([0.0, 0.0])
lp_mid = np.array([
    np.log(gmm["weights"][j]) - 0.5 * np.sum(np.log(2.0 * np.pi * gmm["variances"][j]) + ((x_mid - gmm["means"][j]) ** 2) / gmm["variances"][j])
    for j in range(2)
])
resp_mid = np.exp(lp_mid - np.max(lp_mid))
resp_mid /= np.sum(resp_mid)

assert np.allclose(np.sum(resp, axis=1), 1.0, atol=1e-14)
assert np.isclose(np.sum(gmm["weights"]), 1.0, atol=1e-14)
assert np.mean(np.max(resp, axis=1)) > 0.99
assert np.allclose(resp_mid, [0.5, 0.5], atol=1e-12)
print("Fitted phase means:\n", np.round(gmm["means"], 4))
print("Bulk phase #1 sample 0 responsibilities :", np.round(resp[0], 6))
print("Midpoint coexistence responsibilities   :", np.round(resp_mid, 6), "(Exact 50/50 phase boundary)")
heading("Demonstration 2: Shannon Phase-Assignment Entropy Across a Two-Phase Tie-Line")
eps = 1e-15
shannon_entropy = -np.sum(resp * np.log(np.maximum(resp, eps)), axis=1)
bulk_entropy = float(np.mean(shannon_entropy))
boundary_entropy = float(-np.sum(resp_mid * np.log(resp_mid)))

print(f"{'Tie-Line Coordinate s':>21s} | {'P(Phase 1)':>12s} | {'P(Phase 2)':>12s} | {'Shannon Entropy H [nats]':>24s}")
print("-" * 78)
for s_coord in [-1.0, -0.5, -0.1, 0.0, 0.1, 0.5, 1.0]:
    x_pt = s_coord * np.array([1.5, 1.0])
    lp = np.array([
        np.log(gmm["weights"][j]) - 0.5 * np.sum(np.log(2.0 * np.pi * gmm["variances"][j]) + ((x_pt - gmm["means"][j]) ** 2) / gmm["variances"][j])
        for j in range(2)
    ])
    g_pt = np.exp(lp - np.max(lp))
    g_pt /= np.sum(g_pt)
    h_pt = float(-np.sum(g_pt * np.log(np.maximum(g_pt, eps))))
    print(f"{s_coord:21.2f} | {g_pt[0]:12.6f} | {g_pt[1]:12.6f} | {h_pt:24.6f}")
assert boundary_entropy > 20.0 * bulk_entropy and np.isclose(boundary_entropy, np.log(2.0), atol=1e-12)
heading("Demonstration 3: Three-Phase Triple-Point Coexistence & Gibbs Phase Rule Bound")
t_circ = np.linspace(0.0, 2.0 * np.pi, 30, endpoint=False)
circle_pts = 0.35 * np.column_stack([np.cos(t_circ), np.sin(t_circ)])
c1_tp, c2_tp, c3_tp = np.array([-2.0, 0.0]), np.array([2.0, 0.0]), np.array([0.0, 2.0 * np.sqrt(3.0)])
tp_centroid = (c1_tp + c2_tp + c3_tp) / 3.0
X_tp = np.vstack([circle_pts + c1_tp, circle_pts + c2_tp, circle_pts + c3_tp])

gmm3 = clustering_phases.gmm_em_fit(X_tp, k=3, seed=42)
lp_tp = np.array([
    np.log(gmm3["weights"][j]) - 0.5 * np.sum(np.log(2.0 * np.pi * gmm3["variances"][j]) + ((tp_centroid - gmm3["means"][j]) ** 2) / gmm3["variances"][j])
    for j in range(3)
])
r_tp = np.exp(lp_tp - np.max(lp_tp))
r_tp /= np.sum(r_tp)
ent_tp = float(-np.sum(r_tp * np.log(r_tp)))
print(f"Triple-point coordinates (T*, P*): {np.round(tp_centroid, 4)}")
print(f"Triple-point responsibilities    : {np.round(r_tp, 4)} (Exact 1/3, 1/3, 1/3 invariant point)")
print(f"Triple-point Shannon entropy H   : {ent_tp:.6f} nats == ln(3) = {np.log(3.0):.6f} nats")
assert np.allclose(r_tp, [1.0 / 3.0, 1.0 / 3.0, 1.0 / 3.0], atol=1e-2) and np.isclose(ent_tp, np.log(3.0), atol=1e-3)
heading("Demonstration 4: Bayesian Information Criterion (BIC) & Gibbs Phase Rule Model Selection")
def gibbs_max_coexisting_phases(n_components_c: int, fixed_vars: int = 0) -> int:
    """Maximum coexisting phases P_max = C + 2 - fixed_vars from F = C - P + 2 - fixed_vars >= 0."""
    return n_components_c + 2 - fixed_vars

assert gibbs_max_coexisting_phases(1) == 3 and gibbs_max_coexisting_phases(2, fixed_vars=1) == 3
N_tp, d_tp = X_tp.shape
print(f"{'Components K':>12s} | {'Log-Likelihood l':>16s} | {'Free Params p_K':>15s} | {'BIC(K) (Lower=Better)':>21s} | {'Gibbs Admissible (C=1)':>22s}")
print("-" * 96)
bics = {}
for k_cand in [1, 2, 3, 4, 5]:
    g_k = clustering_phases.gmm_em_fit(X_tp, k=k_cand, seed=42)
    p_k = k_cand * (2 * d_tp + 1) - 1
    bic_k = -2.0 * g_k["log_likelihood"] + p_k * np.log(N_tp)
    bics[k_cand] = bic_k
    gibbs_ok = k_cand <= gibbs_max_coexisting_phases(1)
    print(f"{k_cand:12d} | {g_k['log_likelihood']:16.3f} | {p_k:15d} | {bic_k:21.3f} | {str(gibbs_ok):>22s}")
assert min(bics, key=bics.get) == 3
heading("Inside the Algorithm: By-Hand EM Responsibility E-Step vs. the Library")
log_p = np.zeros((len(X_all), 2))
for j in range(2):
    vj = gmm["variances"][j]
    log_p[:, j] = np.log(gmm["weights"][j]) - 0.5 * (np.sum(np.log(2.0 * np.pi * vj)) + np.sum(((X_all - gmm["means"][j]) ** 2) / vj, axis=1))
m_lp = np.max(log_p, axis=1, keepdims=True)
e_lp = np.exp(log_p - m_lp)
resp_hand = e_lp / np.sum(e_lp, axis=1, keepdims=True)
diff_gmm = float(np.max(np.abs(resp_hand - resp)))
print("by hand (pure NumPy) resp[0]:", np.round(resp_hand[0], 8))
print("library (engml_2_applied)      resp[0]:", np.round(resp[0], 8))
print(f"Max responsibility discrepancy: {diff_gmm:.2e}")
assert diff_gmm < 1e-14
sys.stdout = _stdout
print("Set-up complete: the variables of the lecture's demonstrations are defined.")

Set-up complete: the variables of the lecture's demonstrations are defined.


## Exercise 1

**Problem.** Verify **monotone log-likelihood ascent** $\ell(\theta^{(t+1)}) \ge \ell(\theta^{(t)})$ of the Expectation–Maximization algorithm: run `gmm_em_fit` on `X_tp` ($K=3$) across increasing iteration budgets `max_iter` $\in \{1, 2, 3, 5, 10, 25\}$ and confirm that the log-likelihood is monotonically non-decreasing.

In [2]:
heading("Exercise 1: Monotone EM Log-Likelihood Ascent Verification")
lls = []
for m_it in [1, 2, 3, 5, 10, 25]:
    ll_val = clustering_phases.gmm_em_fit(X_tp, k=3, max_iter=m_it, tol=0.0, seed=42)["log_likelihood"]
    lls.append(ll_val)
    print(f"EM iteration {m_it:2d} | Log-likelihood l(theta) = {ll_val:.6f}")
assert all(lls[i] <= lls[i + 1] + 1e-12 for i in range(len(lls) - 1))


Exercise 1: Monotone EM Log-Likelihood Ascent Verification
EM iteration  1 | Log-likelihood l(theta) = -102.944552
EM iteration  2 | Log-likelihood l(theta) = -102.932813
EM iteration  3 | Log-likelihood l(theta) = -102.932813
EM iteration  5 | Log-likelihood l(theta) = -102.932813
EM iteration 10 | Log-likelihood l(theta) = -102.932813
EM iteration 25 | Log-likelihood l(theta) = -102.932813


**Discussion.** By Jensen's inequality and the non-negativity of Kullback–Leibler divergence, every EM step increases or preserves the observed-data log-likelihood $\ell(\theta^{(t+1)}) \ge \ell(\theta^{(t)})$.

---

## Exercise 2

**Problem.** Audit the **Gibbs phase rule violation** in an unconstrained 5-cluster GMM fit (`k=5`) on a pure substance ($C = 1$): show that if an engineer queries the centroid of a spurious 4-way cluster boundary, 4 clusters appear to coexist simultaneously ($P_{\mathrm{local}} = 4 > C + 2 = 3$), implying a physically impossible negative degree of freedom $F = 1 - 4 + 2 = -1 < 0$.

In [3]:
heading("Exercise 2: Gibbs Phase Rule Violation Audit (F = C - P + 2 < 0)")
C_pure = 1
P_spurious = 4
F_dof = C_pure - P_spurious + 2
print(f"Pure substance (C={C_pure}) with P={P_spurious} coexisting phases -> Gibbs DOF F = C - P + 2 = {F_dof} < 0 (thermodynamically impossible)")
assert F_dof < 0 and P_spurious > gibbs_max_coexisting_phases(C_pure)


Exercise 2: Gibbs Phase Rule Violation Audit (F = C - P + 2 < 0)
Pure substance (C=1) with P=4 coexisting phases -> Gibbs DOF F = C - P + 2 = -1 < 0 (thermodynamically impossible)


**Discussion.** At thermodynamic equilibrium, the chemical potentials $\mu_i^{(\alpha)}(T, P) = \mu_i^{(\beta)}(T, P)$ impose $C(P-1)$ constraints on $2 + P(C-1)$ intensive variables. When $P > C + 2$, the system of equations is overdetermined ($F < 0$).

---

## Exercise 3

**Problem.** Implement a function `active_phase_boundary_Acquisition(gmm_res: dict, X_grid: np.ndarray) -> dict` that evaluates the GMM posterior responsibilities $\boldsymbol{\gamma}(\mathbf{x})$ and Shannon entropy $H(\mathbf{x})$ across a candidate grid `X_grid` and returns the state point $\mathbf{x}^* = \arg\max_{\mathbf{x} \in \mathbf{X}_{\mathrm{grid}}} H(\mathbf{x})$ with highest phase-coexistence uncertainty. Verify on a 2-D grid spanning $[-2, 2]^2$ that it pinpoints the triple point `tp_centroid` of `gmm3`.

In [4]:
heading("Exercise 3: Implement Maximum-Entropy Phase-Boundary Active-Learning Acquisition")
def active_phase_boundary_acquisition(gmm_res: dict, X_grid: np.ndarray) -> dict:
    """Rank candidate (T, P) experiments by GMM Shannon phase-boundary entropy H(x)."""
    Xg = np.atleast_2d(np.asarray(X_grid, dtype=float))
    K_g = len(gmm_res["weights"])
    lp = np.zeros((len(Xg), K_g))
    for j in range(K_g):
        vj = gmm_res["variances"][j]
        lp[:, j] = np.log(gmm_res["weights"][j]) - 0.5 * (np.sum(np.log(2.0 * np.pi * vj)) + np.sum(((Xg - gmm_res["means"][j]) ** 2) / vj, axis=1))
    m_lp = np.max(lp, axis=1, keepdims=True)
    gam = np.exp(lp - m_lp)
    gam /= np.sum(gam, axis=1, keepdims=True)
    ent = -np.sum(gam * np.log(np.maximum(gam, 1e-15)), axis=1)
    best_i = int(np.argmax(ent))
    return {"x_best": Xg[best_i], "max_entropy": float(ent[best_i]), "responsibilities_best": gam[best_i]}

g1 = np.linspace(-1.5, 1.5, 41)
g2 = np.linspace(0.0, 2.5, 51)
G1, G2 = np.meshgrid(g1, g2)
X_cand_grid = np.column_stack([G1.ravel(), G2.ravel(),])
X_cand_grid = np.vstack([X_cand_grid, tp_centroid[None, :]])
acq = active_phase_boundary_acquisition(gmm3, X_cand_grid)
print("Highest-entropy recommended experiment x*:", np.round(acq["x_best"], 4), "(True triple point:", np.round(tp_centroid, 4), ")")
print(f"Max Shannon entropy H(x*) = {acq['max_entropy']:.4f} nats | Responsibilities = {np.round(acq['responsibilities_best'], 4)}")
assert np.allclose(acq["x_best"], tp_centroid, atol=1e-2) and np.isclose(acq["max_entropy"], np.log(3.0), atol=1e-3)


Exercise 3: Implement Maximum-Entropy Phase-Boundary Active-Learning Acquisition
Highest-entropy recommended experiment x*: [0.     1.1547] (True triple point: [0.     1.1547] )
Max Shannon entropy H(x*) = 1.0986 nats | Responsibilities = [0.3333 0.3333 0.3333]


**Discussion.** Maximizing GMM posterior Shannon entropy $H(\mathbf{x})$ points to where the next measurement is most informative: boundaries between two clusters ($H = \ln 2$) and points where three meet ($H = \ln 3$).